<a href="https://colab.research.google.com/github/sathkumara-smna/258739K/blob/main/5GPower.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [185]:
import pandas as pd
# Raw GitHub file URL
#url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/5G_01_03_2026%20-%20Lite.xlsx"
url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/5G%20Raw%20Data.xlsx"

# Load Excel file
df = pd.read_excel(url)

# Show first few rows
df.head()

,Site_ID,Cell_ID,Sector_ID,trigger_ID,datetime,traffic_load_mbps,Rectifier Reading (W)
0,101,1011,1,1,2026-03-01 00:00,61.20,53.10
1,101,1011,1,2,2026-03-01 00:15,67.31,53.24
2,101,1011,1,3,2026-03-01 00:30,72.87,53.37
3,101,1011,1,4,2026-03-01 00:45,76.15,53.45
4,101,1011,1,5,2026-03-01 01:00,81.49,53.57


In [186]:
# Raw GitHub file URL
url1 = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/5G%20Traffic%20Bands.xlsx"

# Load Excel file
info_5g_traffic = pd.read_excel(url1)

# Show first few rows
info_5g_traffic.head()

,Lower,Upper,UpperValue,g1a_power,bbu_extra_power
0,0,60,60,5,2
1,60,200,200,10,4
2,200,400,400,15,6
3,400,800,800,20,8
4,800,1200,1200,25,10


In [187]:
# Create new column and fill with 0
df['aau_count'] = 0
df['bbu_base_power'] = 25
df['g1a_base_power'] = 26.67
df['bbu_traffic_band'] = 0
df['bbu_maxdata_band'] = 0
df['bbu_extra_power'] = 0
df['g1a_extra_band'] = 0
df['g1a_extra_power'] = 0
df['5g_sec_power'] = 0
# Verify
df.head()

,Site_ID,Cell_ID,Sector_ID,trigger_ID,datetime,traffic_load_mbps,Rectifier Reading (W),aau_count,bbu_base_power,g1a_base_power,bbu_traffic_band,bbu_maxdata_band,bbu_extra_power,g1a_extra_band,g1a_extra_power,5g_sec_power
0,101,1011,1,1,2026-03-01 00:00,61.20,53.10,0,25,26.67,0,0,0,0,0,0
1,101,1011,1,2,2026-03-01 00:15,67.31,53.24,0,25,26.67,0,0,0,0,0,0
2,101,1011,1,3,2026-03-01 00:30,72.87,53.37,0,25,26.67,0,0,0,0,0,0
3,101,1011,1,4,2026-03-01 00:45,76.15,53.45,0,25,26.67,0,0,0,0,0,0
4,101,1011,1,5,2026-03-01 01:00,81.49,53.57,0,25,26.67,0,0,0,0,0,0


In [188]:
# Calculate number of unique Cell_IDs under each Site_ID
aau_counts = df.groupby('Site_ID')['Cell_ID'].nunique()

# Map the counts back to dataframe
df['aau_count'] = df['Site_ID'].map(aau_counts)

# Verify
df[['Site_ID', 'Cell_ID', 'aau_count']].head()

,Site_ID,Cell_ID,aau_count
0,101,1011,3
1,101,1011,3
2,101,1011,3
3,101,1011,3
4,101,1011,3


In [189]:
def get_bbu_extra_power(traffic):

    match = info_5g_traffic[
        (traffic >= info_5g_traffic['Lower']) &
        (traffic < info_5g_traffic['Upper'])
    ]

    if not match.empty:
        return match.iloc[0]['bbu_extra_power']

    # fallback for very high traffic
    return info_5g_traffic['bbu_extra_power'].max()

# Apply mapping
df['bbu_traffic_band'] = df['traffic_load_mbps'].apply(get_bbu_extra_power)

# Verify
df[['traffic_load_mbps', 'bbu_traffic_band']].head(5)

,traffic_load_mbps,bbu_traffic_band
0,61.20,4
1,67.31,4
2,72.87,4
3,76.15,4
4,81.49,4


In [190]:
# -------------------------------------------------
# Map UpperValue from info_5g_traffic
# based on traffic_load_mbps range
# into df['bbu_maxdata_band']
# -------------------------------------------------

def get_upper_value(traffic):

    match = info_5g_traffic[
        (traffic >= info_5g_traffic['Lower']) &
        (traffic <= info_5g_traffic['Upper'])
    ]

    if not match.empty:
        return match.iloc[0]['UpperValue']

    # fallback for traffic above highest range
    return info_5g_traffic['UpperValue'].max()

# Apply mapping
df['bbu_maxdata_band'] = df['traffic_load_mbps'].apply(get_upper_value)

# Verify
df[['traffic_load_mbps', 'bbu_maxdata_band']].head(5)

,traffic_load_mbps,bbu_maxdata_band
0,61.20,200
1,67.31,200
2,72.87,200
3,76.15,200
4,81.49,200


In [191]:
# Calculate bbu_extra_power

df['bbu_extra_power'] = round(
    (
        (df['traffic_load_mbps'] / df['bbu_maxdata_band']) *
        df['bbu_traffic_band']
    ) / df['aau_count'],
    2
)

# Verify
df[[
    'traffic_load_mbps',
    'bbu_maxdata_band',
    'bbu_traffic_band',
    'aau_count',
    'bbu_extra_power'
]].head(5)

,traffic_load_mbps,bbu_maxdata_band,bbu_traffic_band,aau_count,bbu_extra_power
0,61.20,200,4,3,0.41
1,67.31,200,4,3,0.45
2,72.87,200,4,3,0.49
3,76.15,200,4,3,0.51
4,81.49,200,4,3,0.54


In [192]:
# -------------------------------------------------
# Map g1a_power from info_5g_traffic
# based on traffic_load_mbps range
# into df['g1a_extra_band']
# -------------------------------------------------

def get_g1a_power(traffic):

    match = info_5g_traffic[
        (traffic >= info_5g_traffic['Lower']) &
        (traffic <= info_5g_traffic['Upper'])
    ]

    if not match.empty:
        return match.iloc[0]['g1a_power']

    # fallback for traffic above highest range
    return info_5g_traffic['g1a_power'].max()

# Apply mapping
df['g1a_extra_band'] = df['traffic_load_mbps'].apply(get_g1a_power)

# Verify
df[['traffic_load_mbps', 'g1a_extra_band']].head(5)

,traffic_load_mbps,g1a_extra_band
0,61.20,10
1,67.31,10
2,72.87,10
3,76.15,10
4,81.49,10


In [193]:
# Calculate g1a_extra_power

df['g1a_extra_power'] = round(
    (
        (df['traffic_load_mbps'] * df['g1a_extra_band'])
        / df['bbu_maxdata_band']
    ) / df['aau_count'],
    2
)

# Verify
df[[
    'traffic_load_mbps',
    'g1a_extra_band',
    'bbu_maxdata_band',
    'aau_count',
    'g1a_extra_power'
]].head(5)

,traffic_load_mbps,g1a_extra_band,bbu_maxdata_band,aau_count,g1a_extra_power
0,61.20,10,200,3,1.02
1,67.31,10,200,3,1.12
2,72.87,10,200,3,1.21
3,76.15,10,200,3,1.27
4,81.49,10,200,3,1.36


In [194]:
df.columns

Index(['Site_ID', 'Cell_ID', 'Sector_ID', 'trigger_ID', 'datetime',
       'traffic_load_mbps', 'Rectifier Reading (W)', 'aau_count',
       'bbu_base_power', 'g1a_base_power', 'bbu_traffic_band',
       'bbu_maxdata_band', 'bbu_extra_power', 'g1a_extra_band',
       'g1a_extra_power', '5g_sec_power'],
      dtype='object')

In [195]:
# Calculate 5g_sec_power

df['5g_sec_power'] = round(
    df['bbu_base_power'] +
    df['g1a_base_power'] +
    df['bbu_extra_power'] +
    df['g1a_extra_power'],
    2
)

# Verify
df[[
    'bbu_base_power',
    'g1a_base_power',
    'bbu_extra_power',
    'g1a_extra_power',
    '5g_sec_power'
]].head(5)

,bbu_base_power,g1a_base_power,bbu_extra_power,g1a_extra_power,5g_sec_power
0,25,26.67,0.41,1.02,53.10
1,25,26.67,0.45,1.12,53.24
2,25,26.67,0.49,1.21,53.37
3,25,26.67,0.51,1.27,53.45
4,25,26.67,0.54,1.36,53.57


In [196]:
df.to_excel("trafficband.xlsx", index=False)

In [197]:
# Create new table: max_cell_Power_5g

max_cell_Power_5g = (
    df.groupby(['Site_ID', 'Cell_ID'])['5g_sec_power']
    .max()
    .reset_index()
)

# Rename column
max_cell_Power_5g = max_cell_Power_5g.rename(
    columns={'5g_sec_power': 'Cell_Max_Power'}
)

# Verify
max_cell_Power_5g.head()

,Site_ID,Cell_ID,Cell_Max_Power
0,101,1011,64.63
1,101,1012,70.11
2,101,1013,72.00
3,103,1031,70.91
4,103,1032,69.42


In [198]:
max_cell_Power_5g.to_excel("max_cell_Power_5g.xlsx", index=False)

In [199]:
# Create new table: max_power_site_5g

max_power_site_5g = (
    max_cell_Power_5g.groupby('Site_ID')['Cell_Max_Power']
    .sum()
    .reset_index()
)

# Rename column
max_power_site_5g = max_power_site_5g.rename(
    columns={'Cell_Max_Power': 'Max_5g_Power'}
)

# Verify
max_power_site_5g.head()

,Site_ID,Max_5g_Power
0,101,206.74
1,103,210.82
2,105,208.33
3,108,211.94
4,110,210.61


In [200]:
max_power_site_5g.to_excel("max_power_site_5g.xlsx", index=False)